In [4]:
import torch

Get Patches from an image given the patch indices. 

Patches are non-overlapping

Case A: 2D images

In [6]:
x = torch.arange(64).reshape(8,8)
x

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7],
        [ 8,  9, 10, 11, 12, 13, 14, 15],
        [16, 17, 18, 19, 20, 21, 22, 23],
        [24, 25, 26, 27, 28, 29, 30, 31],
        [32, 33, 34, 35, 36, 37, 38, 39],
        [40, 41, 42, 43, 44, 45, 46, 47],
        [48, 49, 50, 51, 52, 53, 54, 55],
        [56, 57, 58, 59, 60, 61, 62, 63]])

In [25]:
#break image into 2x2 patches
patch_size = 2
patches_2 = x.unfold(0,patch_size,patch_size).unfold(1,patch_size,patch_size).contiguous().view(-1,2,2)
#break image into 2x2 patches
patch_size = 4
patches_4 = x.unfold(0,patch_size,patch_size).unfold(1,patch_size,patch_size).contiguous().view(-1,4,4)

In [28]:
x

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7],
        [ 8,  9, 10, 11, 12, 13, 14, 15],
        [16, 17, 18, 19, 20, 21, 22, 23],
        [24, 25, 26, 27, 28, 29, 30, 31],
        [32, 33, 34, 35, 36, 37, 38, 39],
        [40, 41, 42, 43, 44, 45, 46, 47],
        [48, 49, 50, 51, 52, 53, 54, 55],
        [56, 57, 58, 59, 60, 61, 62, 63]])

In [29]:
patches_4

tensor([[[ 0,  1,  2,  3],
         [ 8,  9, 10, 11],
         [16, 17, 18, 19],
         [24, 25, 26, 27]],

        [[ 4,  5,  6,  7],
         [12, 13, 14, 15],
         [20, 21, 22, 23],
         [28, 29, 30, 31]],

        [[32, 33, 34, 35],
         [40, 41, 42, 43],
         [48, 49, 50, 51],
         [56, 57, 58, 59]],

        [[36, 37, 38, 39],
         [44, 45, 46, 47],
         [52, 53, 54, 55],
         [60, 61, 62, 63]]])

In [46]:
patches_2

tensor([[[ 0,  1],
         [ 8,  9]],

        [[ 2,  3],
         [10, 11]],

        [[ 4,  5],
         [12, 13]],

        [[ 6,  7],
         [14, 15]],

        [[16, 17],
         [24, 25]],

        [[18, 19],
         [26, 27]],

        [[20, 21],
         [28, 29]],

        [[22, 23],
         [30, 31]],

        [[32, 33],
         [40, 41]],

        [[34, 35],
         [42, 43]],

        [[36, 37],
         [44, 45]],

        [[38, 39],
         [46, 47]],

        [[48, 49],
         [56, 57]],

        [[50, 51],
         [58, 59]],

        [[52, 53],
         [60, 61]],

        [[54, 55],
         [62, 63]]])

In [50]:
x

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7],
        [ 8,  9, 10, 11, 12, 13, 14, 15],
        [16, 17, 18, 19, 20, 21, 22, 23],
        [24, 25, 26, 27, 28, 29, 30, 31],
        [32, 33, 34, 35, 36, 37, 38, 39],
        [40, 41, 42, 43, 44, 45, 46, 47],
        [48, 49, 50, 51, 52, 53, 54, 55],
        [56, 57, 58, 59, 60, 61, 62, 63]])

In [44]:
#get size 4 patches from size 2 patches
W, H = 8, 8
patch_size = 2
W_, H_ = W//patch_size, H//patch_size
#patches_2 shape: (W_*H_, patch_size, patch_size)
patches_2.view(W_, H_, patch_size, patch_size).permute(0,2,1,3).contiguous().view(W_//2, 2*patch_size, H_//2, 2*patch_size).permute(0,2,1,3).contiguous().view(-1,4,4)

tensor([[[ 0,  1,  2,  3],
         [ 8,  9, 10, 11],
         [16, 17, 18, 19],
         [24, 25, 26, 27]],

        [[ 4,  5,  6,  7],
         [12, 13, 14, 15],
         [20, 21, 22, 23],
         [28, 29, 30, 31]],

        [[32, 33, 34, 35],
         [40, 41, 42, 43],
         [48, 49, 50, 51],
         [56, 57, 58, 59]],

        [[36, 37, 38, 39],
         [44, 45, 46, 47],
         [52, 53, 54, 55],
         [60, 61, 62, 63]]])

In [ ]:
def fold_patches_2x(patches, img_shape):
    '''
    increase patch res by 2x
    img_shape: B, C, W, H, D
    Convert a tensor of all patches in img (N, C, P, P, P) to 2x patches (N//8, C, 2P, 2P, 2P)
    N=B*W_*H_*D_
    '''
    B, C, W, H, D = img_shape
    N, C, patch_size, patch_size, patch_size = patches.shape
    W_, H_, D_ = W//patch_size, H//patch_size, D//patch_size
    assert N == B*W_*H_*D_

    return patches.view(B, W_, H_, D_, C, patch_size, patch_size, patch_size) \
        .permute(0, 4, 1, 5, 2, 6, 3, 7).contiguous() \
        .view(B, C, W_//2, 2*patch_size, H_//2, 2*patch_size, D_//2, 2*patch_size) \
        .permute(0 ,2, 4, 6, 1, 3, 5, 7).contiguous() \
        .view(-1, C, 2*patch_size, 2*patch_size,2*patch_size)


#test
B, C, W, H, D = (2,3,128,128,128)
img = torch.randn(B, C, W, H, D)
patch_size = 16
W_, H_, D_ = W//patch_size, H//patch_size, D//patch_size
patches_16 = img.view(B, C, W_, patch_size, H_, patch_size, D_, patch_size).permute(0,2,4,6,1,3,5,7).contiguous().view(-1, C, patch_size, patch_size, patch_size)
target_patch_size = 32
W_, H_, D_ = W//target_patch_size, H//target_patch_size, D//target_patch_size
patches_32 = img.view(B, C, W_, target_patch_size, H_, target_patch_size, D_, target_patch_size).permute(0,2,4,6,1,3,5,7).contiguous().view(-1, C, target_patch_size, target_patch_size, target_patch_size)

torch.all((fold_patches_2x(patches_16, img.shape)) == patches_32)


tensor(True)

In [59]:
def fold_patches(patches, img_shape, up:int=2):
    '''
    increase patch res by up x times
    img_shape: B, C, W, H, D
    Convert a tensor of all patches in img (B*W_*H_*D_, C, P, P, P) to 2x patches (B*W_*H_*D_//8, C, 2P, 2P, 2P)
    W_, H_, D_ are the patch locations
    '''
    B, C, W, H, D = img_shape
    N, C, patch_size, patch_size, patch_size = patches.shape
    W_, H_, D_ = W//patch_size, H//patch_size, D//patch_size
    assert N == B*W_*H_*D_
    
    #target patch size and patch locations
    tgt_patch_size = up*patch_size
    tgt_W_, tgt_H_, tgt_D_  = W_//up, H_//up, D_//up

    return patches.view(B, W_, H_, D_, C, patch_size, patch_size, patch_size) \
        .permute(0, 4, 1, 5, 2, 6, 3, 7).contiguous() \
        .view(B, C, tgt_W_, tgt_patch_size, tgt_H_, tgt_patch_size, tgt_D_, tgt_patch_size) \
        .permute(0, 2, 4, 6, 1, 3, 5, 7).contiguous() \
        .view(-1, C, tgt_patch_size, tgt_patch_size,tgt_patch_size)

#test
B, C, W, H, D = (2,3,128,128,128)
img = torch.randn(B, C, W, H, D)
patch_size = 16
W_, H_, D_ = W//patch_size, H//patch_size, D//patch_size
patches_16 = img.view(B, C, W_, patch_size, H_, patch_size, D_, patch_size).permute(0,2,4,6,1,3,5,7).contiguous().view(-1, C, patch_size, patch_size, patch_size)
target_patch_size = 64
W_, H_, D_ = W//target_patch_size, H//target_patch_size, D//target_patch_size
patches_64 = img.view(B, C, W_, target_patch_size, H_, target_patch_size, D_, target_patch_size).permute(0,2,4,6,1,3,5,7).contiguous().view(-1, C, target_patch_size, target_patch_size, target_patch_size)

torch.all((fold_patches(patches_16, img.shape, up=4)) == patches_64)

tensor(True)

In [49]:
patches_4

tensor([[[ 0,  1,  2,  3],
         [ 8,  9, 10, 11],
         [16, 17, 18, 19],
         [24, 25, 26, 27]],

        [[ 4,  5,  6,  7],
         [12, 13, 14, 15],
         [20, 21, 22, 23],
         [28, 29, 30, 31]],

        [[32, 33, 34, 35],
         [40, 41, 42, 43],
         [48, 49, 50, 51],
         [56, 57, 58, 59]],

        [[36, 37, 38, 39],
         [44, 45, 46, 47],
         [52, 53, 54, 55],
         [60, 61, 62, 63]]])

In [22]:
patches_4

tensor([[[ 0,  1,  2,  3],
         [ 8,  9, 10, 11],
         [16, 17, 18, 19],
         [24, 25, 26, 27]],

        [[ 4,  5,  6,  7],
         [12, 13, 14, 15],
         [20, 21, 22, 23],
         [28, 29, 30, 31]],

        [[32, 33, 34, 35],
         [40, 41, 42, 43],
         [48, 49, 50, 51],
         [56, 57, 58, 59]],

        [[36, 37, 38, 39],
         [44, 45, 46, 47],
         [52, 53, 54, 55],
         [60, 61, 62, 63]]])

In [ ]:
#break image into 4x4 patches
patch_size = 4
x.unfold(0,patch_size,patch_size).unfold(1,patch_size,patch_size)

In [ ]:
patch_idx_h = torch.tensor([0,1])
patch_idx_w = torch.tensor([0,1])

In [ ]:
def get_patches_2D(x:torch.Tensor, patch_idxs:tuple[torch.Tensor], patch_size:int):
    '''Get Patches from an HxW image given the patch indices. 
    Todo: extend it to 3D
    '''
    #x shape = (h, w)
    assert len(x.shape) == 2
    print('image x' , x)
    patch_idx_h, patch_idx_w = patch_idxs
    assert len(patch_idx_h) == len(patch_idx_w)
    print('patch_idx_h, patch_idx_w', patch_idx_h, patch_idx_w)
    print('Num patches = ', len(patch_idx_h))

    #get the h indices of the patches: convert h indices to desired output shape: (num_patches, patch_size, patch_size)
    start_idx_h = patch_idx_h*patch_size
    print('start_idx_h', start_idx_h)
    idxs_h = start_idx_h.repeat(patch_size,1) + torch.arange(patch_size)[:,None]
    print('patch idxs_h', idxs_h)
    idxs_h = torch.repeat_interleave(idxs_h, patch_size, dim=1)
    print('idxs_h repeated for same patch', idxs_h)
    idxs_h = idxs_h.unfold(0,patch_size, patch_size).unfold(1,patch_size, patch_size).squeeze(dim=0)
    print('idxs_h', idxs_h)
    print('idxs_h.shape', idxs_h.shape)

    print('\n')

    #get the w indices of the patches: convert h indices to desired output shape: (num_patches, patch_size, patch_size)
    start_idx_w = patch_idx_w*patch_size
    print('start_idx_w', start_idx_w)
    idxs_w = torch.transpose(start_idx_w.repeat(patch_size,1)+torch.arange(patch_size)[:,None], 0, 1)
    print('patch idxs_w', idxs_w)
    idxs_w = torch.repeat_interleave(idxs_w, patch_size, dim=0)
    print('idxs_w repeated for same patch', idxs_w)
    idxs_w = idxs_w.unfold(0, patch_size, patch_size).unfold(1,patch_size, patch_size).squeeze(dim=1)
    print('idxs_w', idxs_w)
    print('idxs_w.shape', idxs_w.shape)
    
    patches = x[idxs_h,idxs_w]
    print('Selected patches = ', patches)
    print('Selected patches shape = ', patches.shape)
    return patches


patch_idx_h = torch.tensor([0,3,1])
patch_idx_w = torch.tensor([0,2,3])
patch_idxs = (patch_idx_h, patch_idx_w)
patch_size = 2
selected_patches = get_patches_2D(x,patch_idxs, patch_size)



In [ ]:
patches = (x.unfold(0,patch_size,patch_size).unfold(1,patch_size,patch_size)).contiguous().view(-1,2,2)
patches.shape

In [76]:
import itertools
import torch
import numpy as np

def ravel_index(index, shape):
    """Ravel multi-dimensional indices to 1D index
    similar to np.ravel_multi_index
    Args:
        index (torch.tensor): indices in reversed order dn, ..., d1, with shape (..., n)
        shape (tuple): dn, ..., d1
    """
    index = index.to(torch.int64)
    shape = torch.tensor((1,) + shape[::-1], dtype=torch.int64) # =(1, d1, d1*d2, ..., d1*...*dn)
    shape = torch.cumprod(shape, dim=0)[:-1].flip(0) # =(d1*...*dn-1, ..., d1*d2, d1, 1)
    index = (index * shape).sum(dim=-1) # (...,)
    return index



def get_x_patch_idx_from_2x_patch_idx_v1(patch_2x_flat_idx, patch_x_shape, patch_2x_shape):
  assert len(patch_x_shape) == len(patch_2x_shape) in [4, 5]
  ndim = len(patch_x_shape)
  assert np.all(np.array(patch_x_shape[2:]) == 2*np.array(patch_2x_shape[2:]))

  unravled_2x_idxs = torch.stack(torch.unravel_index(patch_2x_flat_idx, patch_2x_shape), dim=0).T
  x_start_offset = torch.tensor([1,1,2,2]) if ndim == 4 else torch.tensor([1,1,2,2,2])
  unravled_start_x_idxs = (unravled_2x_idxs*x_start_offset).to(unravled_2x_idxs)

  offsets = list(itertools.product([0, 1], repeat=ndim-2))
  offsets = [[0,0]+list(offset) for offset in offsets]
  offsets = torch.tensor(offsets).to(unravled_start_x_idxs)

  unravled_all_x_idxs = unravled_start_x_idxs.unsqueeze(1) + offsets
  unravled_all_x_idxs = unravled_all_x_idxs.contiguous().view(-1,len(patch_x_shape))

  flat_x_patch_idx = ravel_index(unravled_all_x_idxs, patch_x_shape)

  return flat_x_patch_idx


def get_x_patch_idx_from_2x_patch_idx_v2(patch_2x_flat_idx, patch_x_shape, patch_2x_shape):  
  assert len(patch_x_shape) == len(patch_2x_shape) in [4, 5]
  ndim = len(patch_x_shape)
  assert np.all(np.array(patch_x_shape[2:]) == 2*np.array(patch_2x_shape[2:]))
  
  num_patches_x = np.prod(patch_x_shape[2:])
  patch_x_idx_arr = torch.arange(num_patches_x)
  if ndim == 4:
    patch_x_idx_arr = patch_x_idx_arr.view(1,1,patch_x_shape[2],patch_x_shape[3])
    #patchify index array to 2x2 patches. a 2x2 patch stores the patch x indices of a 2x patch index
    patch_x_idx_arr = patch_x_idx_arr.view(1, 1, patch_x_shape[2]//2, 2, patch_x_shape[3]//2, 2) \
                                .permute(0, 2, 4, 1, 3, 5).contiguous().view(-1,1,2,2)
  else:
    patch_x_idx_arr = patch_x_idx_arr.view(1,1,patch_x_shape[2],patch_x_shape[3],patch_x_shape[4])
    #reshape index array to 2x2x2. a 2x2x2 patch stores the patch x indices of a 2x patch index
    patch_x_idx_arr = patch_x_idx_arr.view(1, 1, patch_x_shape[2]//2, 2, patch_x_shape[3]//2, 2, patch_x_shape[4]//2, 2) \
                                .permute(0, 2, 4, 6, 1, 3, 5, 7).contiguous().view(-1,1,2,2,2)
  
  patch_x_flat_idx = patch_x_idx_arr[patch_2x_flat_idx].flatten()
  return patch_x_flat_idx

#test
import time

x_16 = torch.arange(512).view(1,1,8,8,8)
x_32 = torch.arange(64).view(1,1,4,4,4)
sampled_idxs_32 = torch.multinomial(torch.ones_like(x_32.flatten()).to(torch.float),num_samples=32)
print('sampled_idxs_32', sampled_idxs_32)

start_time = time.time()
patch_x_from_2x_flat_idx_v1 = get_x_patch_idx_from_2x_patch_idx_v1(sampled_idxs_32,x_16.shape,x_32.shape)
print(f'v1 took {time.time() - start_time} secs')
start_time = time.time()
patch_x_from_2x_flat_idx_v2 = get_x_patch_idx_from_2x_patch_idx_v2(sampled_idxs_32,x_16.shape,x_32.shape)
print(f'v2 took {time.time() - start_time} secs')

# v2 is faster for smaller arrays but slower for larger

assert torch.all(patch_x_from_2x_flat_idx_v1==patch_x_from_2x_flat_idx_v2)


sampled_idxs_32 tensor([41, 17,  7, 58,  0, 61, 42, 47, 29, 60, 24, 59, 53, 19, 46, 21, 11, 22,
         5, 45, 34, 48, 20, 38, 12, 31, 14,  2, 30, 13, 51, 37])
v1 took 0.0004107952117919922 secs
v2 took 0.00014662742614746094 secs


In [78]:
def aggregate_common_x_and_2x_patches(img_patches_x, flat_patch_indices_x, patch_x_shape, img_patches_2x, flat_patch_indices_2x, patch_2x_shape, agg='mean'):
    '''
    Given a set of size x patches and size 2x patches corresponding to the same image,
    aggregate the common regions
    '''
    assert len(patch_x_shape) == len(patch_2x_shape) in [4, 5]
    assert np.all(np.array(patch_x_shape[2:]) == 2*np.array(patch_2x_shape[2:]))
    ndim = len(patch_x_shape)

    flat_patch_indices_x_from_2x = get_x_patch_idx_from_2x_patch_idx_v2(flat_patch_indices_2x, patch_x_shape, patch_2x_shape)

    if ndim == 4:
        N_x,C,x,x = img_patches_x.shape #N_x:num of size x patches
        assert img_patches_2x.shape[1:] == (C,2*x,2*x)
        img_patches_x_from_2x = img_patches_2x.view(-1,C,2,x,2,x).permute(0,2,4,1,3,5).contiguous().view(-1,C,x,x)
    else:
        N_x,C,x,x,x = img_patches_x.shape #N_x:num of size x patches
        assert img_patches_2x.shape[1:] == (C,2*x,2*x,2*x)
        img_patches_x_from_2x = img_patches_2x.view(-1,C,2,x,2,x,2,x).permute(0,2,4,6,1,3,5,7).contiguous().view(-1,C,x,x,x)

    common_patch_indices = (flat_patch_indices_x.unsqueeze(1)==flat_patch_indices_x_from_2x).nonzero()
    common_patches_agg = img_patches_x[common_patch_indices[:,0]] + img_patches_x_from_2x[common_patch_indices[:,1]]
    print('common_patches_agg.shape', common_patches_agg.shape)

    test_common_patch_idxs_  = flat_patch_indices_x[torch.isin(flat_patch_indices_x, flat_patch_indices_x_from_2x)]
    return common_patches_agg, test_common_patch_idxs_


#test
B, C, W, H, D = 2,3,128,128,128
img_x = torch.randn(B, C, W, H, D)
_x = 16
img_patches_x = img_x.view(B,C,W//_x,_x,H//_x,_x,D//_x,_x).permute(0,2,4,6,1,3,5,7).contiguous().view(-1,C,_x,_x,_x)
patch_x_shape = (1, 1, W//_x, H//_x, D//_x)
num_x_patches = (W//_x)*(H//_x)*(D//_x)
sampled_patch_indices_x = torch.multinomial(torch.ones(num_x_patches),num_samples=num_x_patches//2)
sampled_img_patches_x = img_patches_x[sampled_patch_indices_x]

img_2x = torch.rand(B, C, W, H, D)
_2x = 32
img_patches_2x = img_2x.view(2,C,W//_2x,_2x,H//_2x,_2x,D//_2x,_2x).permute(0,2,4,6,1,3,5,7).contiguous().view(-1,C,_2x,_2x,_2x)
patch_2x_shape = (1, 1, W//_2x, H//_2x, D//_2x)
num_2x_patches = (W//_2x)*(H//_2x)*(D//_2x)
sampled_patch_indices_2x = torch.multinomial(torch.ones(num_2x_patches),num_samples=num_2x_patches//2)
sampled_img_patches_2x = img_patches_2x[sampled_patch_indices_2x]

agg_patches, test_common_patch_idxs = aggregate_common_x_and_2x_patches(sampled_img_patches_x, sampled_patch_indices_x, patch_x_shape, \
                        sampled_img_patches_2x, sampled_patch_indices_2x, patch_2x_shape)

test_sum_imgs = (img_x + img_2x).view(B,C,W//_x,_x,H//_x,_x,D//_x,_x).permute(0,2,4,6,1,3,5,7).contiguous().view(-1,C,_x,_x,_x)
assert(torch.all(test_sum_imgs[test_common_patch_idxs] == agg_patches))


common_patches_agg.shape torch.Size([119, 3, 16, 16, 16])


In [109]:
import itertools
import torch

def ravel_index_(index, shape):
    """Ravel multi-dimensional indices to 1D index
    similar to np.ravel_multi_index
    Args:
        index (torch.tensor): indices in reversed order dn, ..., d1, with shape (..., n)
        shape (tuple): dn, ..., d1
    """
    index = torch.tensor(index, dtype=torch.int64)
    shape = torch.tensor((1,) + shape[::-1], dtype=torch.int64) # =(1, d1, d1*d2, ..., d1*...*dn)
    shape = torch.cumprod(shape, dim=0)[:-1].flip(0) # =(d1*...*dn-1, ..., d1*d2, d1, 1)
    index = (index * shape).sum(dim=-1) # (...,)
    return index


def get_16_idx_from_32_flat_idx(flat_idxs_32,x_16_shape,x_32_shape):
  assert len(x_16_shape) == len(x_32_shape)
  print('flat_idxs_32 ', flat_idxs_32)

  unravled_idxs_32 = torch.unravel_index(flat_idxs_32, x_32_shape)
  unravled_idxs_32_T = torch.stack(unravled_idxs_32, dim=0).T
  unravled_idxs_start_16 = unravled_idxs_32_T*torch.tensor([1,1,2,2])
  print(unravled_idxs_start_16.shape)

  list(itertools.product([0, 1], repeat=2))
  offsets = list(itertools.product([0, 1], repeat=2))
  offsets = [[0,0]+list(offset) for offset in offsets]
  offsets = torch.tensor(offsets)
  print('offsets ', offsets)

  unraveled_idxs_all_16 = unravled_idxs_start_16.unsqueeze(1) + offsets
  unraveled_idxs_all_16 = unraveled_idxs_all_16.contiguous().view(-1,len(x_16_shape))

  flat_idxs_16 = ravel_index_(unraveled_idxs_all_16, x_16_shape)
  print('flat_idxs_16 ', flat_idxs_16)

  unraveled_idxs_all_16 = torch.unbind(unraveled_idxs_all_16, dim=1)
  return flat_idxs_16, unraveled_idxs_all_16

x_16 = torch.linspace(0, 63, steps=64).view(1,1,8,8)
x_32 = torch.linspace(0, 15, steps=16).view(1,1,4,4)
sampled_idxs_32 = torch.multinomial(torch.ones_like(x_32.flatten()),num_samples=8)

flat_idxs_16, unraveled_idxs_all_16 = get_16_idx_from_32_flat_idx(sampled_idxs_32,x_16.shape,x_32.shape)
test_raveled_idxs_16 = x_16[unraveled_idxs_all_16]
assert torch.all(flat_idxs_16 == test_raveled_idxs_16)

flat_idxs_32  tensor([ 6, 13,  4, 11, 12, 15,  2,  0])
torch.Size([8, 4])
offsets  tensor([[0, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 1, 1]])
flat_idxs_16  tensor([20, 21, 28, 29, 50, 51, 58, 59, 16, 17, 24, 25, 38, 39, 46, 47, 48, 49,
        56, 57, 54, 55, 62, 63,  4,  5, 12, 13,  0,  1,  8,  9])


/tmp/ca550013/login23-g-1_119198/ipykernel_120080/454033549.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  index = torch.tensor(index, dtype=torch.int64)


In [10]:
def ravel_tuple_index(index:tuple[torch.Tensor], shape:tuple[int]):
    """Ravel multi-dimensional indices to 1D index
    similar to np.ravel_multi_index
    Args:
        index (torch.tensor): indices in reversed order dn, ..., d1, with shape (..., n)
        shape (tuple): dn, ..., d1
    """
    index = torch.stack(index,dim=0).T
    #index = torch.tensor(index, dtype=torch.int64)
    shape = torch.tensor((1,) + shape[::-1], dtype=torch.int64) # =(1, d1, d1*d2, ..., d1*...*dn)
    shape = torch.cumprod(shape, dim=0)[:-1].flip(0) # =(d1*...*dn-1, ..., d1*d2, d1, 1)
    index = (index * shape).sum(dim=-1) # (...,)
    return index